# SIB - Exercício 3.3

Teste da classe `SelectPercentile` com o dataset iris.

In [1]:
import numpy as np

from si.io.csv_file import read_csv
from si.feature_selection.select_percentile import SelectPercentile
from si.statistics.f_classification import f_classification

In [2]:
iris = read_csv('../datasets/iris/iris.csv', sep=',', features=True, label=True)

print('dimensões:', iris.shape())
print('features:', iris.features)

dimensões: (150, 4)
features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


## Valores de F e p por feature

In [3]:
F, p = f_classification(iris)

for feature, f_value, p_value in zip(iris.features, F, p):
    print(f'{feature:15s} F = {f_value:8.2f}   p = {p_value:.2e}')

sepal_length    F =   119.26   p = 1.67e-31
sepal_width     F =    47.36   p = 1.33e-16
petal_length    F =  1179.03   p = 3.05e-91
petal_width     F =   959.32   p = 4.38e-85


As features das pétalas têm valores de F muito mais altos do que as das sépalas, ou seja, separam melhor as três espécies. É por isso que serão as escolhidas.

## percentile = 50 (metade das features)

In [4]:
selector = SelectPercentile(score_func=f_classification, percentile=50)
selector.fit(iris)

print('F estimado:', np.round(selector.F, 2))

iris_50 = selector.transform(iris)
print('dimensões:', iris_50.shape())
print('features selecionadas:', iris_50.features)

F estimado: [ 119.26   47.36 1179.03  959.32]
dimensões: (150, 2)
features selecionadas: ['petal_length', 'petal_width']


## percentile = 25 (uma feature) e percentile = 100 (todas)

In [5]:
for percentile in (25, 50, 75, 100):
    new_dataset = SelectPercentile(percentile=percentile).fit_transform(iris)
    print(f'percentile = {percentile:3d} -> {new_dataset.shape()[1]} features: {new_dataset.features}')

percentile =  25 -> 1 features: ['petal_length']
percentile =  50 -> 2 features: ['petal_length', 'petal_width']
percentile =  75 -> 3 features: ['sepal_length', 'petal_length', 'petal_width']
percentile = 100 -> 4 features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


O número de features selecionadas é `int(n_features * percentile / 100)`. Com 4 features, o percentile 25 dá 1 feature, o 50 dá 2 e o 75 dá 3.

## Tratamento de empates

Exemplo do enunciado: 10 features, `percentile=40`, com o valor 5.6 repetido três vezes.

In [6]:
from si.data.dataset import Dataset

F_exemplo = np.array([1.2, 3.4, 2.1, 5.6, 4.3, 5.6, 7.8, 6.5, 5.6, 3.2])
dataset_exemplo = Dataset(X=np.tile(F_exemplo, (3, 1)), y=np.array([0, 1, 0]))

selector = SelectPercentile(percentile=40)
selector.F = F_exemplo
selector.p = np.zeros(10)
selector._is_fitted = True

print('threshold (percentil 60 dos F):', np.percentile(F_exemplo, 60))
print('features acima do threshold:', F_exemplo[F_exemplo > np.percentile(F_exemplo, 60)])

resultado = selector.transform(dataset_exemplo)
print('features finais:', resultado.X[0])
print('número de features:', resultado.shape()[1])

threshold (percentil 60 dos F): 5.6
features acima do threshold: [7.8 6.5]
features finais: [5.6 5.6 7.8 6.5]
número de features: 4


40% de 10 features são 4 features. O threshold é 5.6 e só duas features o ultrapassam estritamente (7.8 e 6.5). As duas em falta vêm dos empates no próprio threshold, o que devolve `[5.6, 5.6, 7.8, 6.5]`, tal como o enunciado descreve.